# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rufatj/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Before picking a rule I checked two signals separately, each on its own bucket table, because I
didn't want to bake in a signal that just sounds right.

**Signal 1 - staleness.** My first guess was "old + not updated in a while = declining." Split
straight on `days_since_last_update >= 180` (n=174 stale vs n=29,826 not-stale), the stale group
actually declines *less* than the rest (47.1% vs 54.2%) - the opposite of what I expected.
Verdict on the raw signal: **OPPOSITE**. But before dropping it I checked why, and the reason is
obvious once you look: most of those 174 stale pages have basically no traffic left (median
impressions_90d = 16), so there's nothing left to decline. Once I also require the page to still
be visible (`impressions_90d >= 500`, only 17 rows), the picture flips hard: 16 of those 17
declined (94.1%), way above the 54.2% base rate. So my real verdict is **MIXED** - staleness alone
is not a signal, staleness *and* visibility together is a strong one, and that's exactly why
`stale_visible_page` is written as an AND condition and not a single column check.

**Signal 2 - CTR vs. position.** Grouped mean CTR by `position_tier` (impressions >= 100 so the
means aren't noise): page_1 0.355%, top_3 0.334%, striking 0.256%, page_3_5 0.142%, deep 0.055%.
That's roughly a 6x range from top to bottom, so "low CTR" only means something once you know what
tier a page is even competing in - a 0.15% CTR is unremarkable in `deep` but a real miss in
`page_1`. Verdict: **CONFIRMED**.

**My rule, in plain words:** flag a page if either (a) it's stale AND still visible
(`days_since_last_update >= 180` and `impressions_90d >= 500`) - the strongest signal I found, or
(b) it's visible with a real position (`impressions_90d >= 500`, has a known position tier) and its
own CTR sits below half of what its *own tier* normally gets - a real, tier-adjusted gap, not a
flat number. Everything else is left alone for now.

**Reason codes this rule can output:** `stale_visible_page` (condition a, takes priority when both
fire since it's the rarer, stronger signal), `low_ctr_visible_page` (condition b only), or
`no_flag` (neither fired - action `monitor`).


In [1]:
import os

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "run this from inside the repo"

import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Signal 1: staleness, three-way bucket table with n ---
stale_visible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
stale_invisible = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] < 500)
not_stale = df["days_since_last_update"] < 180

print("Signal 1 - staleness bucket table:")
for name, mask in [
    ("stale & visible (>=500 impr)", stale_visible),
    ("stale & low-visibility (<500 impr)", stale_invisible),
    ("not stale (<180d)", not_stale),
]:
    n = mask.sum()
    rate = df.loc[mask, "trend_direction"].eq("down").mean()
    print(f"  {name:38} n={n:>6,}  decline_rate={rate:.3f}")
print(f"  {'base rate (all pages)':38} n={len(df):>6,}  decline_rate={df['trend_direction'].eq('down').mean():.3f}")

# --- Signal 2: CTR by position tier, bucket table with n ---
visible100 = df[df["impressions_90d"] >= 100]
tiers = visible100.groupby("position_tier")["ctr"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print("\nSignal 2 - CTR by position tier:")
print(tiers.round(4).to_string())


Signal 1 - staleness bucket table:
  stale & visible (>=500 impr)           n=    17  decline_rate=0.941
  stale & low-visibility (<500 impr)     n=   157  decline_rate=0.420
  not stale (<180d)                      n=29,826  decline_rate=0.542
  base rate (all pages)                  n=30,000  decline_rate=0.542

Signal 2 - CTR by position tier:
                 mean  count
position_tier               
page_1         0.3548   8633
top_3          0.3341    533
striking       0.2558   5903
page_3_5       0.1424   6058
deep           0.0554    879


## 2. Build the ranked queue (writes the CSV)

The score is deliberately readable, no fitted weights: `stale_visible_page` rows get a fixed high
score (1000 + impressions_90d/1000, so bigger pages break ties), `low_ctr_visible_page` rows get
scored by how big their CTR gap is relative to their tier, scaled by log(impressions) so a big gap
on a bigger page ranks above the same gap on a tiny page. Everything else scores 0 and gets
`monitor`. The ranked queue is written to `work/outputs/baseline_action_score.csv` (gitignored, it
regenerates every run); the metrics that back it up get saved to a small JSON that I do commit.


In [2]:
has_tier = df["position_tier"] != "no_data"
tier_means = df.loc[df["impressions_90d"] >= 100].groupby("position_tier")["ctr"].mean()
df["tier_mean_ctr"] = df["position_tier"].map(tier_means)

visible = df["impressions_90d"] >= 500
ctr_gap = visible & has_tier & (df["ctr"] < 0.5 * df["tier_mean_ctr"])
ctr_gap_only = ctr_gap & ~stale_visible  # stale_visible_page takes priority on overlap

df["baseline_action_score"] = np.select(
    [stale_visible, ctr_gap_only],
    [1000 + df["impressions_90d"] / 1000, (df["tier_mean_ctr"] - df["ctr"]) * np.log1p(df["impressions_90d"])],
    default=0.0,
)
df["reason_code"] = np.select([stale_visible, ctr_gap_only], ["stale_visible_page", "low_ctr_visible_page"], default="no_flag")
df["action"] = np.select([stale_visible, ctr_gap_only], ["refresh", "review_ctr"], default="monitor")
df["is_declining_label"] = df["trend_direction"].eq("down").astype(int)  # eval label only, never a feature

ranked = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "baseline_action_score", "reason_code", "action",
            "impressions_90d", "ctr", "avg_position", "position_tier", "days_since_last_update"]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"wrote work/outputs/baseline_action_score.csv ({len(ranked):,} rows)")

flagged = ranked["reason_code"] != "no_flag"
print(f"flagged: {flagged.sum():,} ({flagged.mean():.1%} of pages) -> "
      f"decline rate {ranked.loc[flagged, 'is_declining_label'].mean():.3f} "
      f"vs {ranked.loc[~flagged, 'is_declining_label'].mean():.3f} for the rest "
      f"(base rate {ranked['is_declining_label'].mean():.3f})")

def precision_at_k(labels, k):
    return labels[:k].mean()

metrics = {
    "base_rate": float(ranked["is_declining_label"].mean()),
    "n_rows": int(len(ranked)),
    "n_flagged": int(flagged.sum()),
    "precision_at_10": float(precision_at_k(ranked["is_declining_label"].values, 10)),
    "precision_at_50": float(precision_at_k(ranked["is_declining_label"].values, 50)),
    "precision_at_100": float(precision_at_k(ranked["is_declining_label"].values, 100)),
}
for k in (10, 50, 100):
    print(f"precision@{k}: {metrics[f'precision_at_{k}']:.3f}  (base rate {metrics['base_rate']:.3f})")

import json
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/w04_baseline_metrics.json")


wrote work/outputs/baseline_action_score.csv (30,000 rows)
flagged: 6,781 (22.6% of pages) -> decline rate 0.655 vs 0.509 for the rest (base rate 0.542)
precision@10: 1.000  (base rate 0.542)
precision@50: 0.740  (base rate 0.542)
precision@100: 0.670  (base rate 0.542)
wrote work/outputs/w04_baseline_metrics.json


## 3. Top-10 review

Reading my own top ten by hand, like the skill says - this is where bad logic shows itself.


In [3]:
top10 = ranked.head(10)
for i, row in top10.reset_index(drop=True).iterrows():
    wrong_if = (
        "the traffic drop is really consolidation to a sibling page or seasonality, not staleness"
        if row["reason_code"] == "stale_visible_page"
        else "the low CTR is because the query intent doesn't match the page, not a fixable title/meta issue"
    )
    print(f"{i+1}. action={row['action']}  reason={row['reason_code']}  "
          f"score={row['baseline_action_score']:.1f}  impressions_90d={row['impressions_90d']:,}  "
          f"declined={'yes' if row['is_declining_label'] else 'no'}")
    print(f"    would be wrong if: {wrong_if}")


1. action=refresh  reason=stale_visible_page  score=1061.7  impressions_90d=61,678  declined=yes
    would be wrong if: the traffic drop is really consolidation to a sibling page or seasonality, not staleness
2. action=refresh  reason=stale_visible_page  score=1059.5  impressions_90d=59,472  declined=yes
    would be wrong if: the traffic drop is really consolidation to a sibling page or seasonality, not staleness
3. action=refresh  reason=stale_visible_page  score=1025.7  impressions_90d=25,715  declined=yes
    would be wrong if: the traffic drop is really consolidation to a sibling page or seasonality, not staleness
4. action=refresh  reason=stale_visible_page  score=1013.3  impressions_90d=13,299  declined=yes
    would be wrong if: the traffic drop is really consolidation to a sibling page or seasonality, not staleness
5. action=refresh  reason=stale_visible_page  score=1007.8  impressions_90d=7,812  declined=yes
    would be wrong if: the traffic drop is really consolidation to a

## 4. Weak picks + leakage check

**Weak pick:** all ten of my top ten come from a single client. Only 17 rows satisfy
`stale_visible_page` in the whole 30,000-row slice, and they happen to cluster almost entirely on
one client that apparently has a batch of old pages nobody revisited. My score has no per-client
cap, so one client's backlog crowds out every other client's `low_ctr_visible_page` candidates from
the top of the list. A production version would need a per-client limit (or normalize the score
within client) so the queue actually serves every client, not just the one with the most stale
pages.

**Leakage check:** the score only uses `days_since_last_update`, `impressions_90d`, `ctr`,
`avg_position` / `position_tier` - all knowable at the moment I'd decide whether to review a page,
none of them computed from a future window. `trend_direction` (and the `trend_pct` it's built
from) is used only to build `is_declining_label` for evaluating the rule after the fact, checked
below to confirm it never entered the score itself. No FlyRank product flags
(`health_score`, `priority_score`, `action_type`) exist in this dataset in the first place, so
there's nothing to accidentally rebuild and feed back in.


In [4]:
score_inputs = {"days_since_last_update", "impressions_90d", "ctr", "avg_position", "position_tier"}
label_derived = {"trend_direction", "trend_pct", "is_declining_label"}
print("score inputs used:", sorted(score_inputs))
print("overlap with label-derived columns:", score_inputs & label_derived, "(must be empty)")
assert not (score_inputs & label_derived), "leakage: a label-derived column ended up in the score"

print()
print("client concentration in the top 10:")
print(top10["client_id"].value_counts().to_string())


score inputs used: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d', 'position_tier']
overlap with label-derived columns: set() (must be empty)

client concentration in the top 10:
client_id
client_7f2253d7e2    10


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.
